In [1]:
import pandas as pd
from random import random
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import math
import joblib
np.random.seed(43)

In [2]:
team_data = pd.read_csv('test_data_2024.csv')
modeling_data = pd.read_csv('modeling_data.csv')


In [3]:
model = joblib.load('xgb_calibrated.pkl')
label_encoder = joblib.load('label_encoder.pkl')

/Users/BenTucker/opt/anaconda3/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [10:21:29] WARNING: /Users/runner/work/xgboost/xgboost/src/gbm/../common/error_msg.h:80: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  warnings.warn(smsg, UserWarning)
/Users/BenTucker/opt/anaconda3/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator IsotonicRegression from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintain

In [4]:
def predict_drive_outcome(game_state, team_ratings):
    offense = 'home' if game_state['isHomeOffense'] else 'away'
    features = pd.DataFrame([{
        'startYardline': game_state['startYardline'], # Metadata suggests startYardsToGoal
        'startPeriod': game_state['startPeriod'],
        'game_minutes_remaining': game_state['gameMinutesRemaining'],
        'startScoreDiff': game_state['startScoreDiff'],
        'isHomeOffense': game_state['isHomeOffense'],
        
        'ppa_overall_diff': team_ratings[f"{offense}_ppa_overall_diff"],
        'ppa_passing_diff': team_ratings[f"{offense}_ppa_passing_diff"],
        'ppa_rushing_diff': team_ratings[f"{offense}_ppa_rushing_diff"],
        'ppa_third_down_diff': team_ratings[f"{offense}_ppa_third_down_diff"],
        'rating_diff': team_ratings[f"{offense}_rating_diff"],
        'elo_diff': team_ratings[f"{offense}_elo_diff"]
    }])
    
    probs = model.predict_proba(features)[0]
    outcome_index = np.random.choice(len(probs), p=probs)
    result_label = label_encoder.inverse_transform([outcome_index])[0]
    
    return result_label

In [5]:
modeling_data = modeling_data[modeling_data['startPeriod'].isin([1,2,3,4])]
modeling_data = modeling_data[modeling_data['season']==2024]
modeling_data['initial_yards_to_go']=np.where(modeling_data['isHomeOffense'],100-modeling_data['startYardline'],modeling_data['startYardline'])
modeling_data['final_yards_to_go']=modeling_data['initial_yards_to_go']-modeling_data['yards']
modeling_data = modeling_data.sort_values(['gameId','driveNumber'])
modeling_data['minutes_elapsed'] = modeling_data.groupby('gameId')['game_minutes_remaining'].diff(-1).abs()
modeling_data['minutes_elapsed'] = modeling_data['minutes_elapsed'].fillna(modeling_data['game_minutes_remaining'])
modeling_data = modeling_data[modeling_data['minutes_elapsed']>0]

In [108]:
def record_drive_statistics(drive_result, game_state, game_statistics):
    new_startYardline = 0
    points_scored = 0
    def_points = 0
    minutes_elapsed = 0
    yards_gained = 0
    punt_yards = 42
    regular_possession_change = True
    isHomeOffense = game_state['isHomeOffense']
    startYardline = game_state['startYardline']
    game_minutes_remaining = game_state['gameMinutesRemaining']
    
    if drive_result == 'TD':
        points_scored = 7
        if isHomeOffense:
            yards_gained = 100 - startYardline
            new_startYardline = 75
        else:
            yards_gained = startYardline
            new_startYardline = 25
        
    if drive_result == 'FG':
        points_scored = 3
        if isHomeOffense:
            yards_gained = (100-startYardline) - np.random.choice(modeling_data[(modeling_data['drive_result']==drive_result) & (modeling_data['final_yards_to_go']<=(105-startYardline))]['final_yards_to_go'].values)
            new_startYardline = 75
        else:
            yards_gained = startYardline - np.random.choice(modeling_data[(modeling_data['drive_result']==drive_result) & (modeling_data['final_yards_to_go']<=(5+startYardline))]['final_yards_to_go'].values)
            new_startYardline = 25
            
    if drive_result == 'MISSED_FG':
        points_scored = 0
        if isHomeOffense:
            yards_gained = (100-startYardline) - np.random.choice(modeling_data[(modeling_data['drive_result']==drive_result) & (modeling_data['final_yards_to_go']<=(105-startYardline))]['final_yards_to_go'].values)
            new_startYardline = 100 - (100-startYardline-yards_gained)
        else:
            yards_gained = startYardline - np.random.choice(modeling_data[(modeling_data['drive_result']==drive_result) & (modeling_data['final_yards_to_go']<=(5+startYardline))]['final_yards_to_go'].values)
            new_startYardline = startYardline-yards_gained
    
    if drive_result == 'PUNT':
        points_scored = 0
        if isHomeOffense:
            yards_gained = (100-startYardline) - np.random.choice(modeling_data[(modeling_data['drive_result']==drive_result) & (modeling_data['final_yards_to_go']<=(105-startYardline))]['final_yards_to_go'].values)
            new_startYardline = 70
        else:
            yards_gained = startYardline - np.random.choice(modeling_data[(modeling_data['drive_result']==drive_result) & (modeling_data['final_yards_to_go']<=(5+startYardline))]['final_yards_to_go'].values)
            new_startYardline = 30
    
    if drive_result == 'TURNOVER':
        points_scored = 0
        if isHomeOffense:
            yards_gained = (100-startYardline) - np.random.choice(modeling_data[(modeling_data['drive_result']==drive_result) & (modeling_data['final_yards_to_go']<=(105-startYardline))]['final_yards_to_go'].values)
            new_startYardline = 50
        else:
            yards_gained = startYardline - np.random.choice(modeling_data[(modeling_data['drive_result']==drive_result) & (modeling_data['final_yards_to_go']<=(5+startYardline))]['final_yards_to_go'].values)
            new_startYardline = 50
            
    if drive_result == 'DOWNS':
        points_scored = 0
        if isHomeOffense:
            yards_gained = (100-startYardline) - np.random.choice(modeling_data[(modeling_data['drive_result']==drive_result) & (modeling_data['final_yards_to_go']<=(105-startYardline))]['final_yards_to_go'].values)
            new_startYardline = 100 - (100-startYardline-yards_gained)
        else:
            yards_gained = startYardline - np.random.choice(modeling_data[(modeling_data['drive_result']==drive_result) & (modeling_data['final_yards_to_go']<=(5+startYardline))]['final_yards_to_go'].values)
            new_startYardline = startYardline-yards_gained
        
    if drive_result == 'SAFETY':
        if isHomeOffense:
            yards_gained = -1 * startYardline
            new_startYardline = 75
        else:
            yards_gained = -1 * (100-startYardline)
            new_startYardline = 75
        def_points = 2
            
    if drive_result == 'TURNOVER_TD':
        regular_possession_change = False
        def_points = 7
        if isHomeOffense:
            yards_gained = (100-startYardline) - np.random.choice(modeling_data[(modeling_data['drive_result']==drive_result) & (modeling_data['final_yards_to_go']<=(105-startYardline))]['final_yards_to_go'].values)
            new_startYardline = 25
        else:
            yards_gained = startYardline - np.random.choice(modeling_data[(modeling_data['drive_result']==drive_result) & (modeling_data['final_yards_to_go']<=(5+startYardline))]['final_yards_to_go'].values)
            new_startYardline = 75
            
    minutes_elapsed = np.random.choice(np.append(modeling_data[modeling_data['yards'].between(yards_gained-5,yards_gained+5)]['minutes_elapsed'].values,1))
    
    if game_state['gameMinutesRemaining'] < minutes_elapsed:
        points_scored = 0
        def_points = 0
        yards_gained = round(game_state['gameMinutesRemaining']/minutes_elapsed,0)
        minutes_elapsed = game_state['gameMinutesRemaining']
        drive_result = 'END_OF_HALF'
        
    if (game_minutes_remaining > 30) & (game_state['gameMinutesRemaining'] - minutes_elapsed < 30):
        points_scored = 0
        def_points = 0
        yards_gained = round(yards_gained*(game_state['gameMinutesRemaining']-30)/minutes_elapsed,0)
        minutes_elapsed = game_state['gameMinutesRemaining'] - 30
        drive_result = 'END_OF_HALF'
    
    game_statistics['home_yards']+= yards_gained if game_state['isHomeOffense'] else 0
    game_statistics['home_score']+= points_scored if game_state['isHomeOffense'] else def_points
    game_statistics['away_yards']+= 0 if game_state['isHomeOffense'] else yards_gained
    game_statistics['away_score']+= def_points if game_state['isHomeOffense'] else points_scored
    
    
    game_state['startYardline']=new_startYardline
    game_state['gameMinutesRemaining']=game_state['gameMinutesRemaining'] - minutes_elapsed
    game_state['startPeriod']=math.floor((60-game_state['gameMinutesRemaining'])/15)+1
    game_state['isHomeOffense']= game_state['isHomeOffense'] if drive_result in ['END_OF_HALF','TURNOVER_TD'] else not game_state['isHomeOffense']
    game_state['startScoreDiff']= (game_statistics['home_score']-game_statistics['away_score'])*(2*game_state['isHomeOffense']-1)
    
    
    if game_state['gameMinutesRemaining']==30:
        game_state['isHomeOffense'] = not game_state['isHomeStarting']
    
    drive_info = {
        'gameId':game_statistics['gameId'],
        'driveId':game_state['driveCount'],
        'isHomeOffense':isHomeOffense,
        'result':drive_result,
        'yards':yards_gained,
        'minutes_elapsed':minutes_elapsed
    }
    
    game_state['driveCount']+=1
    
    return game_state, drive_info
        

In [109]:
def simulate_game(game_id, home_team, away_team):
    rng = np.random.default_rng()
    coin_toss = rng.uniform(0,1)
    offense, defense = (home_team, away_team) if coin_toss > 0.5 else (away_team, home_team)
    receiving_team = offense #record which team gets the ball to start each half
    deferring_team = defense
    
    drive_log = []
    
    #initializing the team_ratings used in predicting drive outcomes
    team_ratings = record_team_ratings(home_team, away_team) 
    
    #initializing the game_state metrics, used in predicting drive outcomes
    game_state = {
        'isHomeStarting':offense==home_team,
        'isHomeOffense': offense==home_team,
        'startScoreDiff': 0,
        'gameMinutesRemaining':60,
        'startPeriod':1,
        'startYardline': 25 if offense==home_team else 75,
        'driveCount':1
    }
    
    #initializing game_statistics, used to track data across games and drives
    game_statistics = {
        'gameId':game_id,
        'home_team':home_team,
        'home_yards':0,
        'home_score':0,
        'away_team':away_team,
        'away_yards':0,
        'away_score':0
    }
    
    #while game_state['gameMinutesRemaining']>0:
    
    while game_state['gameMinutesRemaining'] > 0:
        drive_result = predict_drive_outcome(game_state, team_ratings)
        game_state, drive_info = record_drive_statistics(drive_result, game_state, game_statistics)
        drive_log.append(drive_info)
        
            
        
        
        #drive_data, next_off, next_def = simulate_drive(game_id, drive_id, offense, defense)
        #game_log.append(drive_data)
        #final_scores[offense] += drive_data['points_scored']
        #offense, defense = next_off, next_def
        
    
    return drive_log, game_statistics

In [110]:
def record_team_ratings(home_team, away_team):
    team_ratings = {
        'home_ppa_overall_diff':team_data.loc[team_data['team'] == home_team, 'ppa_off_overall'].values[0] + team_data.loc[team_data['team'] == away_team, 'ppa_def_overall'].values[0],
        'away_ppa_overall_diff' : team_data.loc[team_data['team'] == away_team, 'ppa_off_overall'].values[0] + team_data.loc[team_data['team'] == home_team, 'ppa_def_overall'].values[0],
        'home_ppa_passing_diff' : team_data.loc[team_data['team'] == home_team, 'ppa_off_passing'].values[0] + team_data.loc[team_data['team'] == away_team, 'ppa_def_passing'].values[0],
        'away_ppa_passing_diff' : team_data.loc[team_data['team'] == away_team, 'ppa_off_passing'].values[0] + team_data.loc[team_data['team'] == home_team, 'ppa_def_passing'].values[0],
        'home_ppa_rushing_diff' : team_data.loc[team_data['team'] == home_team, 'ppa_off_rushing'].values[0] + team_data.loc[team_data['team'] == away_team, 'ppa_def_rushing'].values[0],
        'away_ppa_rushing_diff' : team_data.loc[team_data['team'] == away_team, 'ppa_off_rushing'].values[0] + team_data.loc[team_data['team'] == home_team, 'ppa_def_rushing'].values[0],
        'home_ppa_third_down_diff' : team_data.loc[team_data['team'] == home_team, 'ppa_off_third_down'].values[0] + team_data.loc[team_data['team'] == away_team, 'ppa_def_third_down'].values[0],
        'away_ppa_third_down_diff' : team_data.loc[team_data['team'] == away_team, 'ppa_off_third_down'].values[0] + team_data.loc[team_data['team'] == home_team, 'ppa_def_third_down'].values[0],
        'home_rating_diff' : team_data.loc[team_data['team']==home_team,'off_rating'].values[0] - team_data.loc[team_data['team']==away_team,'def_rating'].values[0],
        'away_rating_diff' : team_data.loc[team_data['team']==away_team,'off_rating'].values[0] - team_data.loc[team_data['team']==home_team,'def_rating'].values[0],
        'home_elo_diff' : team_data.loc[team_data['team']==home_team,'off_elo'].values[0] - team_data.loc[team_data['team']==away_team,'def_elo'].values[0],
        'away_elo_diff' : team_data.loc[team_data['team']==away_team,'off_elo'].values[0] - team_data.loc[team_data['team']==home_team,'def_elo'].values[0],
    }
    return team_ratings

In [112]:
def sim_games(games, home_team, away_team):
    drives = []
    game_summaries = []
    for game_id in range(1,games+1):
        print(game_id)
        drive_logs, game_statistics = simulate_game(game_id, home_team, away_team)
        drives.extend(drive_logs)
        summary = {
            'game_id': game_id,
            'home_team':home_team,
            'away_team':away_team,
            'home_score':game_statistics['home_score'],
            'away_score':game_statistics['away_score'],
            'home_yards':game_statistics['home_yards'],
            'away_yards':game_statistics['away_yards'],
            'winner': "Tie" if game_statistics['home_score'] == game_statistics['away_score'] else home_team if game_statistics['home_score']>game_statistics['away_score'] else away_team
        }
        game_summaries.append(summary)
    
    df_drives = pd.DataFrame(drives)
    df_results = pd.DataFrame(game_summaries)
    return df_drives, df_results

In [107]:
drive_results, game_results = sim_games(50, 'Georgia', 'Tennessee')

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50


In [70]:
drive_results.to_csv('simulated_drives.csv',index=False)
game_results.to_csv('simulated_games.csv',index=False)